In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('data/bitcoin_tweet.parquet')

### Data Analysis

In [3]:
df.describe()

,user_followers,user_friends,account_age_days,url_count
count,632179.0,632179.0,632179.000000,632179.000000
mean,4223.438121,768.950281,2555.759374,0.000321
std,53956.704763,2686.274589,1558.663658,0.017917
min,0.0,0.0,786.000000,0.000000
25%,40.0,93.0,1359.000000,0.000000
50%,163.0,270.0,1769.000000,0.000000
75%,786.0,742.0,3721.000000,0.000000
max,8076992.0,656204.0,6924.000000,1.000000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 632179 entries, 16 to 4595426
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   user_name         632179 non-null  object             
 1   user_created      632179 non-null  datetime64[us, UTC]
 2   user_followers    632179 non-null  Int64              
 3   user_friends      632179 non-null  Int64              
 4   user_verified     632179 non-null  boolean            
 5   date              632179 non-null  datetime64[us, UTC]
 6   text              632179 non-null  object             
 7   hashtags          632178 non-null  object             
 8   mentions          632179 non-null  object             
 9   account_age_days  632179 non-null  int64              
 10  age_bucket        632179 non-null  category           
 11  url_count         632179 non-null  int64              
 12  text_clean        632179 non-null  object      

### Numerical column analysis

In [5]:
numerical_columns = df.select_dtypes(include=['int64', 'Int64']).columns.tolist()

In [6]:
df[numerical_columns].describe().loc[['min', 'max']].T

,min,max
user_followers,0.0,8076992.0
user_friends,0.0,656204.0
account_age_days,786.0,6924.0
url_count,0.0,1.0


- `account_age_days` has been pre processed (filtered by account age date to remove the new accounts) 
- `url_count` is engineered column that is grabbed from the text column (it has been capped at 1 to remove the **spam, ad** tweets)

#### Categorical column analysis

In [7]:
cat_columns = df.select_dtypes(include=['category', 'object']).columns.tolist()
cat_columns.remove('text')
cat_columns.remove('text_clean') # will analyse it later
cat_columns

['user_name', 'hashtags', 'mentions', 'age_bucket']

In [8]:
df.hashtags.str[0]

16             BTC
24             BTC
25             BTC
31         Bitcoin
51          cronje
            ...   
4595407       fiat
4595411    Bitcoin
4595422    bitcoin
4595425     Solana
4595426    Bitcoin
Name: hashtags, Length: 632179, dtype: object

In [9]:
df.age_bucket.value_counts()

age_bucket
3-5 yr     279370
>10 yr     162001
5-10 yr    144117
2-3 yr      46691
<2 yr           0
Name: count, dtype: int64

`account_age` with less than **2 years** has been removed during date pre-processing

### `hashtags` Analysis

In [10]:
hashtags_exploded = df['hashtags'].explode().reset_index(drop=True)
hashtag_counts = hashtags_exploded.str.lower().value_counts().reset_index()

print(f'total unique hashtags: {len(hashtag_counts):,}')

more_than_once = hashtag_counts[hashtag_counts['count'] > 1]
print(f'Total unique hashtags that have appeared more than once: {len(more_than_once):,}')
print("\nTop hashtags appearing more than once:")
hashtag_counts.head()

total unique hashtags: 34,164
Total unique hashtags that have appeared more than once: 12,020

Top hashtags appearing more than once:


,hashtags,count
0,bitcoin,465471
1,btc,233680
2,crypto,51301
3,eth,27724
4,ethereum,20270


From ~600k tweets there are 34k unique hashtags found

### `user_name` Analysis

In [11]:
username_counts_df = df.user_name.value_counts()
username_counts_df.head(5)

user_name
Herry Mason                    99
Asher V.                       98
Crypto Advisor                 96
crypto spot                    95
500 headlines about bitcoin    94
Name: count, dtype: int64

During the pre processing we have removed accounts with **more than 100 tweets** (Bot account tweet, daily price news bot tweet)

In [12]:
for text in df[df.user_name == '500 headlines about bitcoin']['text'].head():
    print(text)

You might not see the need for money that can’t be counterfeited. But 1.4 billion Indians do. #bitcoin
The system is rigged. So we’re building one that can’t be. #bitcoin
Make your last overdraft your last. #bitcoin
Feel the rush of finally getting to agree with Marc Andreessen. #bitcoin
There’s no such thing as a blood bitcoin. #bitcoin


Some of the account with high tweet counts still seem to a bot account but we cannot remove the account with tweet volume cap

In [13]:
print(f"number of unique users: {df.user_name.nunique():,}")
print(df.user_name.value_counts().describe().loc[['min', '75%', 'max']].to_frame().T)

number of unique users: 227,205
       min  75%   max
count  1.0  2.0  99.0


most of the users (75%) only tweeted twice

In [14]:
print(f"95th percentile of users tweeted {df.user_name.value_counts().quantile(0.95)} times")

95th percentile of users tweeted 10.0 times


In [15]:
for text in df[df.user_name.isin(username_counts_df[username_counts_df == 2].index)]['text'].head(5):
    print(text)

@iamDonaldYusuf @yuzomausman @aproko_doctor Not now as there is fud of sub $30k #BTC ...people no wan risk buy at dz prices imo
Move Digital Energy Through Cyber Space. #Bitcoin #BTC
@MWCunfiltered If we went to 25K-27K rallied to 32k and chopped down back again that would be awful if you’re holding.  But maybe alts destroyed and #BTC at such an attractive price is what’s needed
Somebody please take #BTC behind the shed and put it out of its misery.
@TTGMaverick I will examine your theory of being highs on Wednesday. Unfortunately, #btc is now coincide with the movement of #SPX.


most of them are **replied** or **mentioned** someone of their tweet. which is great for **social graph** also this means **all human/non-bot** tweets which is great for sentiment analysis

#### `Mentions` exploration

In [16]:
mentions_exploded = df.mentions.explode()

In [17]:
mentions_exploded.value_counts().head()

mentions
elonmusk           11821
saylor              9472
peterschiff         7564
cz_binance          4817
bitcoinmagazine     4743
Name: count, dtype: int64

there are 2 people mentioned the most (other than elonmusk). might be useful for social graph

In [18]:
for text in df[(df.user_name =='saylor')]['text'].head():
    print(text)

i told you all yesterday to buy the #Bitcoin  dip

did you do it?

probably not... 

BUT there is still hope for you

you can still mortgage your house and buy bitcoin

you can still sell your clothes and buy bitcoin

you can still live on the poverty line... and buy bitcoin
@smolleader @smolEggOpinion this is the correct way of going about it, if you have anything else left to sell I suggest doing that and buying more #Bitcoin
they say that smols have topped

they said the same thing about #Bitcoin in 2011, 2013 and 2017 

i think im noticing a pattern here
i really think a16z should just buy #bitcoin with leverage and just send it
looking for a job after the events of today if anyone needs a good boat captain im available to steer the ship #Bitcoin


He seems to be very active in twitter in bitcoin community

**peterschiff** (american stockbroker) and **elon musk** and the other 2 did not appear in the clean or the initial dataset\
so `saylor` is probably the most popular/communicative one in bitcoin community

In [19]:
df.text

16         @MexcResearch @MEXC_Fans @MEXC_Updates #BTC - ...
24         @iamDonaldYusuf @yuzomausman @aproko_doctor No...
25         I personally ignore weekend moves and chill wi...
31         Move Digital Energy Through Cyber Space. #Bitc...
51         1/2 End of shitcoin season, lasted even longer...
                                 ...                        
4595407    @MartiniGuyYT All you need is an internet conn...
4595411    @lexfridman @saylor 1. How can #Bitcoin become...
4595422    @jkenney Now eliminate the debt and add #bitco...
4595425    @TheMoonCarl #Solana #MATIC maybe #Ada but if ...
4595426    Whatever the mainstream media is narrating and...
Name: text, Length: 632179, dtype: object